# 04 — LSTM Transaction Sequence Anomaly Detection

Input: sequence per wallet [(value, gas, time_delta), ...] panjang T=64.

Arsitektur: 2x LSTM autoencoder → reconstruction error = anomaly score.

⚠️ Training di CPU ~30 min untuk 10k wallet. GPU ~3 min.

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Model definition
class LstmAutoencoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.output = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        latent = h_n[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder_lstm(latent)
        return self.output(out)

model = LstmAutoencoder().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Synthetic sequence data (ganti dengan sequence wallet nyata)
np.random.seed(42)
n_wallets, seq_len, input_dim = 500, 64, 3
normal_seq = np.random.randn(n_wallets, seq_len, input_dim).astype(np.float32) * 0.5
# Inject beberapa anomaly
anomaly_idx = np.random.choice(n_wallets, 25, replace=False)
normal_seq[anomaly_idx] *= np.random.uniform(3, 6, size=(25, 1, 1))
print(f'Sequences: {normal_seq.shape}')

In [ ]:
# Train
dataset = TensorDataset(torch.from_numpy(normal_seq))
loader = DataLoader(dataset, batch_size=32, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
model.train()
losses = []
for epoch in range(50):
    epoch_loss = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    losses.append(epoch_loss / len(dataset))
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 loss={losses[-1]:.4f}')

In [ ]:
plt.plot(losses); plt.title('Training Loss'); plt.xlabel('Epoch'); plt.show()

In [ ]:
# Evaluate — reconstruction error per wallet
model.eval()
with torch.no_grad():
    all_data = torch.from_numpy(normal_seq).to(device)
    recon = model(all_data)
    errors = ((recon - all_data) ** 2).mean(dim=(1, 2)).cpu().numpy()

threshold = np.percentile(errors, 95)
detected = errors >= threshold
print(f'Threshold (95th): {threshold:.4f}')
print(f'Detected anomalies: {detected.sum()}/{n_wallets}')
print(f'True anomalies: {len(anomaly_idx)}')
print(f'Recall: {detected[anomaly_idx].sum()}/{len(anomaly_idx)}')